# H6 · `ui/charts.py`

## What this file is for

Eight charts as inline SVG strings -- no chart library, no build step, nothing the engine's
zero-dependency policy would have to make an exception for. Each answers one question:
`coverage_grid` how narrow the testing claim is, `agreement_over_time` whether a regression is
visible in the run it appeared, `response_curve` whether a curve kinks where a lookup was missed,
`usa_map` and `verdict` the one-screen answers, `premium_spread` and `slot_bars` for the layered
programme.

**Colours are stated once, and agreement is blue, not green.** Green reads as *"good"*, and a match
is not a virtue -- it is the expected case. What deserves attention is grey (nothing asked) and amber
(nothing moved).

**`NOT APPLICABLE` is never inside a failure segment**, in every chart that shows it. Conflating
*"ISO does not offer this here"* with *"we got it wrong"* is this file's easiest lie to tell by
accident, and every function is written to make that conflation require effort.

**Depends on:** nothing in this set -- it draws whatever shape of data it is handed, which is why
[`H5 runstore.py`](05-runstore.ipynb)'s `coverage()`, `history()` and `qa_rollup()` exist: to put a
real run's data into that shape.

## Its public surface

Generated from the module, so it can't drift.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent.parent))

import inspect
from ui import charts as c

for name, obj in vars(c).items():
    if name.startswith("_") or getattr(obj, "__module__", None) != c.__name__:
        continue
    if inspect.isfunction(obj):
        print(f"def {name}{inspect.signature(obj)}")
    elif name.isupper():
        v = repr(obj)
        print(f"{name} = {v if len(v) < 90 else v[:87] + '...'}")

## The smallest thing that works

Every chart is one function returning one SVG string. `IPython.display.SVG` renders it inline when
this runs in Jupyter.

In [ ]:
from IPython.display import SVG, display

svg = c.status_bars({
    "compared": True, "rated": 12, "agree": 10,
    "differ": ["TX"], "premium_only": [], "engine_stopped": [],
    "not_applicable": [], "errors": [],
})
print(f"{len(svg)} characters of SVG, no library involved")
display(SVG(svg))

## The interesting case

### The map is a tile grid, not a projection, and Hawaii is drawn blank on purpose

Every jurisdiction gets the **same size square**. A real projection would draw Texas two hundred
times the area of Rhode Island and say something untrue about where the testing effort went --
each state carries one submission, however large. **Hawaii is drawn and permanently coloured
"not filed"**, rather than left off the grid, because leaving it off would hide that ISO's corpus
does not include it at all.

In [ ]:
status = {"TX": "differs", "CA": "agrees", "NY": "refused", "AK": "partial"}
m = c.usa_map(status, title="A run's outcome by state")
display(SVG(m))
print("HI forced to 'absent' regardless of the status dict:", "HI" not in status)

### The one-screen verdict computes its percentage over *comparable* outcomes only

`not_applicable` and `refused` sit in the bar in their own colours, but the headline percentage is
`agree / (agree + differs)` -- **not** `agree / everything`. Folding *"ISO didn't offer this"* into
the denominator would let a jurisdiction with nothing tested inflate an agreement rate that has
nothing to do with it.

In [ ]:
v = c.verdict(agree=40, differs=3, not_applicable=6, refused=1, uncompared=1)
display(SVG(v))

## What it refuses

A chart with nothing to draw says so in words, in the chart's own space -- not a blank box, and not
an exception. `response_curve` needs two or more points per jurisdiction to draw a line; handed one,
it explains what would fill it instead of guessing or crashing.

In [ ]:
one_point = {"TX": [{"value": "1,000,000 CSL", "ours": "8896"}]}
rc = c.response_curve("occurrence_limit", one_point)
display(SVG(rc))
print("this is `empty()`'s message, not an exception:", "run the same control" in rc)

## Try it yourself

1. `premium_spread` calls out any state at **twice the median** as a tail. Build a `points` list
   with one state deliberately far out and confirm it gets labelled red while the rest stay blue.
2. `slot_bars` draws a bar red when a slot "moved nothing in any state" -- which is either a fact
   about ISO's filing or a deductible the harness never actually applied. How would you tell those
   two apart from the chart alone? (You can't -- that's `qa_review.py`'s job, not this file's.)
3. `_axis_order` sorts `"500,000 CSL"` before `"2,000,000 CSL"` by parsing the leading digits.
   Find a legal value in the corpus that would defeat that parser, and confirm whether one exists.

In [ ]:
# your turn